<a href="https://colab.research.google.com/github/SBethune103/virtual-running-coach-pipeline/blob/dev/notebooks/01_data_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required packages (run once)
!pip install -q kaggle datasets

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import polars as pl

# Use Google Drive so data persists
BASE = Path("/content/drive/MyDrive/virtual-running-coach")
DATA_RAW = BASE / "data/raw"
DATA_PROCESSED = BASE / "data/processed"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("✅ Google Drive mounted")
print("Data will be saved to:", DATA_RAW)

Mounted at /content/drive
✅ Google Drive mounted
Data will be saved to: /content/drive/MyDrive/virtual-running-coach/data/raw


In [ ]:
!pip install -q --upgrade kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 15.5 MB/s eta 0:00:00


In [ ]:
# Upload kaggle.json
from google.colab import files
files.upload()  # Upload your kaggle.json file

# Move kaggle.json to correct location
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [ ]:


# Best current datasets
datasets = [
    "mexwell/long-distance-running-dataset",                    # Large real training data
    "aiaiaidavid/the-big-dataset-of-ultra-marathon-running",   # Ultra results
    "heesoo37/120-years-of-olympic-history-athletes-and-results", # Olympic historical
    "olegoaer/running-races-strava",                           # Strava-style amateur races
    "likithagedipudi/run-club-marathon-performance-dataset",   # Training-to-race
    "beridzeg45/runners-dataset"                               # Track events
]

for ds in datasets:
    try:
        name = ds.split("/")[-1]
        print(f"Downloading {name} ...")
        !kaggle datasets download -d {ds} -p {DATA_RAW} --unzip --quiet
        print(f"✅ {name} downloaded")
    except Exception as e:
        print(f"❌ Failed {name}: {e}")

print("\n✅ Downloads finished")

Dataset URL: https://www.kaggle.com/datasets/mexwell/long-distance-running-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
✅ long-distance-running-dataset downloaded
Dataset URL: https://www.kaggle.com/datasets/aiaiaidavid/the-big-dataset-of-ultra-marathon-running
License(s): CC0-1.0
✅ the-big-dataset-of-ultra-marathon-running downloaded
Dataset URL: https://www.kaggle.com/datasets/heesoo37/120-years-of-olympic-history-athletes-and-results
License(s): CC0-1.0
✅ 120-years-of-olympic-history-athletes-and-results downloaded
Dataset URL: https://www.kaggle.com/datasets/olegoaer/running-races-strava
License(s): CC-BY-NC-SA-4.0
✅ running-races-strava downloaded
Dataset URL: https://www.kaggle.com/datasets/likithagedipudi/run-club-marathon-performance-dataset
License(s): CC0-1.0
✅ run-club-marathon-performance-dataset downloaded
Dataset URL: https://www.kaggle.com/datasets/beridzeg45/runners-dataset
License(s): other
✅ runners-dataset downloaded

✅ Downloads finished


In [ ]:
print("📁 Files in raw data folder:")
for f in sorted(DATA_RAW.glob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1_000_000
        print(f" - {f.name}  ({size_mb:.1f} MB)")


📁 Files in raw data folder:
 - TWO_CENTURIES_OF_UM_RACES  (789.8 MB)
 - TWO_CENTURIES_OF_UM_RACES.csv  (827.8 MB)
 - athlete_events.csv  (41.5 MB)
 - covid-containment-and-health-index.csv  (2.2 MB)
 - covid-stringency-index.csv  (2.2 MB)
 - noc_regions.csv  (0.0 MB)
 - policy_response_indexes.csv  (0.0 MB)
 - raw-data-kaggle.csv  (2.1 MB)
 - run_ww_2019_d.csv  (956.5 MB)
 - run_ww_2019_m.csv  (38.5 MB)
 - run_ww_2019_q.csv  (11.8 MB)
 - run_ww_2019_w.csv  (148.6 MB)
 - run_ww_2020_d.csv  (958.0 MB)
 - run_ww_2020_m.csv  (37.9 MB)
 - run_ww_2020_q.csv  (11.7 MB)
 - run_ww_2020_w.csv  (147.5 MB)
 - runners.csv  (3.7 MB)
 - stay-at-home-covid.csv  (1.9 MB)
 - test.csv  (3.5 MB)
 - train.csv  (14.1 MB)
 - workplace-closures-covid.csv  (1.9 MB)


In [ ]:
print("📁 Files in raw data folder:")
for file in sorted(DATA_RAW.glob("**/*")):
    if file.is_file():
        print(f" - {file.name}  ({file.parent.name if file.parent != DATA_RAW else 'root'})")

print("\n" + "="*60)
print("Quick look at key files:\n")

# === Long-distance training data ===
long_dist_files = list(DATA_RAW.glob("*run_ww*"))
if long_dist_files:
    print("✅ Long-distance training files found:")
    for f in long_dist_files[:6]:
        print("   ", f.name)

    # Load a sample
    sample_file = long_dist_files[0]
    df_sample = pl.read_csv(sample_file, n_rows=5000)
    print(f"\nSample from {sample_file.name}: {len(df_sample):,} rows")
    print(df_sample.head(3))
else:
    print("No long-distance files found")

📁 Files in raw data folder:
 - TWO_CENTURIES_OF_UM_RACES  (root)
 - TWO_CENTURIES_OF_UM_RACES.csv  (root)
 - athlete_events.csv  (root)
 - covid-containment-and-health-index.csv  (root)
 - covid-stringency-index.csv  (root)
 - noc_regions.csv  (root)
 - policy_response_indexes.csv  (root)
 - raw-data-kaggle.csv  (root)
 - run_ww_2019_d.csv  (root)
 - run_ww_2019_m.csv  (root)
 - run_ww_2019_q.csv  (root)
 - run_ww_2019_w.csv  (root)
 - run_ww_2020_d.csv  (root)
 - run_ww_2020_m.csv  (root)
 - run_ww_2020_q.csv  (root)
 - run_ww_2020_w.csv  (root)
 - runners.csv  (root)
 - stay-at-home-covid.csv  (root)
 - test.csv  (root)
 - train.csv  (root)
 - workplace-closures-covid.csv  (root)

Quick look at key files:

✅ Long-distance training files found:
    run_ww_2019_d.csv
    run_ww_2019_m.csv
    run_ww_2019_q.csv
    run_ww_2019_w.csv
    run_ww_2020_d.csv
    run_ww_2020_m.csv

Sample from run_ww_2019_d.csv: 5,000 rows
shape: (3, 9)
┌─────┬────────────┬─────────┬──────────┬───┬────────┬─

In [ ]:
# === Olympic data ===
olympic_file = DATA_RAW / "athlete_events.csv"
if olympic_file.exists():
    try:
        df_olympic = pl.read_csv(
            olympic_file,
            n_rows=10000,
            infer_schema_length=10000,
            null_values=["NA", ""]
        )
        print(f"\n✅ Olympic data: {len(df_olympic):,} rows loaded")
        print("Sample Sports:", df_olympic["Sport"].unique().head(10).to_list())
        print("Sample columns:", df_olympic.columns[:10])
    except Exception as e:
        print("Olympic load error:", e)
else:
    print("Olympic file not found")


✅ Olympic data: 10,000 rows loaded
Sample Sports: ['Rhythmic Gymnastics', 'Football', 'Ice Hockey', 'Synchronized Swimming', 'Biathlon', 'Volleyball', 'Cross Country Skiing', 'Beach Volleyball', 'Speed Skating', 'Badminton']
Sample columns: ['ID', 'Name', 'Sex', 'Age', 'Height', 'Weight', 'Team', 'NOC', 'Games', 'Year']


In [ ]:
# === Ultra Marathon data ===
ultra_file = DATA_RAW / "TWO_CENTURIES_OF_UM_RACES.csv"
if ultra_file.exists():
    try:
        df_ultra = pl.read_csv(
            ultra_file,
            n_rows=5000,
            infer_schema_length=20000,      # Give it more rows to infer types
            ignore_errors=True              # Skip bad rows for now
        )
        print(f"\n✅ Ultra-marathon data: {len(df_ultra):,} rows loaded")
        print("Sample columns:", df_ultra.columns[:10])
        print(df_ultra.head(2))
    except Exception as e:
        print("Ultra load error:", e)
else:
    print("Ultra file not found")


✅ Ultra-marathon data: 5,000 rows loaded
Sample columns: ['Year of event', 'Event dates', 'Event name', 'Event distance/length', 'Event number of finishers', 'Athlete performance', 'Athlete club', 'Athlete country', 'Athlete year of birth', 'Athlete gender']
shape: (2, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Year of   ┆ Event     ┆ Event     ┆ Event dis ┆ … ┆ Athlete   ┆ Athlete   ┆ Athlete   ┆ Athlete  │
│ event     ┆ dates     ┆ name      ┆ tance/len ┆   ┆ gender    ┆ age       ┆ average   ┆ ID       │
│ ---       ┆ ---       ┆ ---       ┆ gth       ┆   ┆ ---       ┆ category  ┆ speed     ┆ ---      │
│ i64       ┆ str       ┆ str       ┆ ---       ┆   ┆ str       ┆ ---       ┆ ---       ┆ i64      │
│           ┆           ┆           ┆ str       ┆   ┆           ┆ str       ┆ str       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2018      ┆ 06.0